In [69]:
!pip -q install -U transformers datasets evaluate rouge_score sentencepiece accelerate nltk

In [70]:
import os
import re
import shutil
import random
import numpy as np
import pandas as pd
import torch
import evaluate
import nltk

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

print("torch:", torch.__version__)
import transformers
print("transformers:", transformers.__version__)

nltk.download("punkt")
set_seed(42)

torch: 2.10.0+cu128
transformers: 5.3.0


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [71]:
import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*Both `max_new_tokens`.*and `max_length`.*"
)

In [72]:
import re

def clean_synopsis(text: str) -> str:
    if not isinstance(text, str):
        return text

    t = text.strip()

    patterns = [
        r"^[A-Z0-9\-/ ]{2,80} reported\s+",
        r"^[A-Z0-9\-/ ]{2,80} Reported\s+",
        r"^[A-Za-z0-9\-/ ,]{2,100} reported\s+",
        r"^[A-Za-z0-9\-/ ,]{2,100} Reported\s+",
    ]

    for p in patterns:
        t_new = re.sub(p, "", t)
        if t_new != t:
            t = t_new
            break

    if len(t) > 0:
        t = t[0].upper() + t[1:]

    return t

def clean_prediction(text: str) -> str:
    if not isinstance(text, str):
        return text

    t = text.strip()

    patterns = [
        r"^[A-Z0-9\-/ ]{2,80} reported\s+",
        r"^[A-Za-z0-9\-/ ,]{2,100} reported\s+",
    ]

    for p in patterns:
        t = re.sub(p, "", t)

    if len(t) > 0:
        t = t[0].upper() + t[1:]

    return t

In [73]:
MODEL_NAME = "google/flan-t5-base"

SOURCE_COL = "Report 1_Narrative"
TARGET_COL = "Report 1.2_Synopsis"

USE_SUBSET = True
TRAIN_SAMPLES = 1500
VALID_SAMPLES = 200
TEST_SAMPLES = 200

MAX_INPUT_LENGTH = 320
MAX_TARGET_LENGTH = 48

NUM_EPOCHS = 1
LR = 3e-4

TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 2
GRAD_ACCUM = 2

WEIGHT_DECAY = 0.01
OUTPUT_DIR = "/kaggle/working/flan_t5_asrs_quick"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [74]:
dataset = load_dataset("elihoole/asrs-aviation-reports")

if "validation" not in dataset and "test" not in dataset:
    split_1 = dataset["train"].train_test_split(test_size=0.2, seed=42)
    split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
    dataset = DatasetDict({
        "train": split_1["train"],
        "validation": split_2["train"],
        "test": split_2["test"],
    })
else:
    dataset = DatasetDict({
        "train": dataset["train"],
        "validation": dataset["validation"] if "validation" in dataset else dataset["test"],
        "test": dataset["test"],
    })

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['acn_num_ACN', 'Time_Date', 'Time.1_Local Time Of Day', 'Place_Locale Reference', 'Place.1_State Reference', 'Place.2_Relative Position.Angle.Radial', 'Place.3_Relative Position.Distance.Nautical Miles', 'Place.4_Altitude.AGL.Single Value', 'Place.5_Altitude.MSL.Single Value', 'Environment_Flight Conditions', 'Environment.1_Weather Elements / Visibility', 'Environment.2_Work Environment Factor', 'Environment.3_Light', 'Environment.4_Ceiling', 'Environment.5_RVR.Single Value', 'Aircraft 1_ATC / Advisory', 'Aircraft 1.1_Aircraft Operator', 'Aircraft 1.2_Make Model Name', 'Aircraft 1.3_Aircraft Zone', 'Aircraft 1.4_Crew Size', 'Aircraft 1.5_Operating Under FAR Part', 'Aircraft 1.6_Flight Plan', 'Aircraft 1.7_Mission', 'Aircraft 1.8_Nav In Use', 'Aircraft 1.9_Flight Phase', 'Aircraft 1.10_Route In Use', 'Aircraft 1.11_Airspace', 'Aircraft 1.12_Maintenance Status.Maintenance Deferred', 'Aircraft 1.13_Maintenance Status.Records Complete',

In [75]:
def is_valid(example):
    src = example.get(SOURCE_COL)
    tgt = example.get(TARGET_COL)
    return (
        isinstance(src, str) and isinstance(tgt, str)
        and len(src.strip()) > 0
        and len(tgt.strip()) > 0
    )

dataset = DatasetDict({
    split: dataset[split].filter(is_valid)
    for split in dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['acn_num_ACN', 'Time_Date', 'Time.1_Local Time Of Day', 'Place_Locale Reference', 'Place.1_State Reference', 'Place.2_Relative Position.Angle.Radial', 'Place.3_Relative Position.Distance.Nautical Miles', 'Place.4_Altitude.AGL.Single Value', 'Place.5_Altitude.MSL.Single Value', 'Environment_Flight Conditions', 'Environment.1_Weather Elements / Visibility', 'Environment.2_Work Environment Factor', 'Environment.3_Light', 'Environment.4_Ceiling', 'Environment.5_RVR.Single Value', 'Aircraft 1_ATC / Advisory', 'Aircraft 1.1_Aircraft Operator', 'Aircraft 1.2_Make Model Name', 'Aircraft 1.3_Aircraft Zone', 'Aircraft 1.4_Crew Size', 'Aircraft 1.5_Operating Under FAR Part', 'Aircraft 1.6_Flight Plan', 'Aircraft 1.7_Mission', 'Aircraft 1.8_Nav In Use', 'Aircraft 1.9_Flight Phase', 'Aircraft 1.10_Route In Use', 'Aircraft 1.11_Airspace', 'Aircraft 1.12_Maintenance Status.Maintenance Deferred', 'Aircraft 1.13_Maintenance Status.Records Complete',

In [76]:
if USE_SUBSET:
    dataset["train"] = dataset["train"].shuffle(seed=42).select(
        range(min(TRAIN_SAMPLES, len(dataset["train"])))
    )
    dataset["validation"] = dataset["validation"].shuffle(seed=42).select(
        range(min(VALID_SAMPLES, len(dataset["validation"])))
    )
    dataset["test"] = dataset["test"].shuffle(seed=42).select(
        range(min(TEST_SAMPLES, len(dataset["test"])))
    )

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['acn_num_ACN', 'Time_Date', 'Time.1_Local Time Of Day', 'Place_Locale Reference', 'Place.1_State Reference', 'Place.2_Relative Position.Angle.Radial', 'Place.3_Relative Position.Distance.Nautical Miles', 'Place.4_Altitude.AGL.Single Value', 'Place.5_Altitude.MSL.Single Value', 'Environment_Flight Conditions', 'Environment.1_Weather Elements / Visibility', 'Environment.2_Work Environment Factor', 'Environment.3_Light', 'Environment.4_Ceiling', 'Environment.5_RVR.Single Value', 'Aircraft 1_ATC / Advisory', 'Aircraft 1.1_Aircraft Operator', 'Aircraft 1.2_Make Model Name', 'Aircraft 1.3_Aircraft Zone', 'Aircraft 1.4_Crew Size', 'Aircraft 1.5_Operating Under FAR Part', 'Aircraft 1.6_Flight Plan', 'Aircraft 1.7_Mission', 'Aircraft 1.8_Nav In Use', 'Aircraft 1.9_Flight Phase', 'Aircraft 1.10_Route In Use', 'Aircraft 1.11_Airspace', 'Aircraft 1.12_Maintenance Status.Maintenance Deferred', 'Aircraft 1.13_Maintenance Status.Records Complete',

In [77]:
def add_clean_target(example):
    example["clean_target"] = clean_synopsis(example[TARGET_COL])
    return example

dataset = DatasetDict({
    split: dataset[split].map(add_clean_target)
    for split in dataset
})

In [78]:
def split_sentences_simple(text):
    text = re.sub(r"\s+", " ", text).strip()
    sents = re.split(r'(?<=[\.\!\?])\s+', text)
    return [s.strip() for s in sents if len(s.strip()) > 0]

def score_sentence(sent):
    s = sent.lower()
    score = 0

    # Événement / gravité
    if "turbulence" in s:
        score += 5
    if "injur" in s or "paramedic" in s:
        score += 8
    if "tcas" in s or "ra" in s or "ta" in s:
        score += 7
    if "conflict" in s or "traffic" in s:
        score += 6
    if "separation" in s:
        score += 7
    if "tower" in s:
        score += 8
    if "without contacting tower" in s:
        score += 12
    if "not handed off" in s or "failed to hand" in s:
        score += 10
    if "failed to follow atc instructions" in s:
        score += 12
    if "vectored" in s or "vector" in s:
        score += 5
    if "off the approach" in s:
        score += 9
    if "localizer" in s or "ils" in s:
        score += 4
    if "requested lower" in s:
        score += 5
    if "did not want lower" in s:
        score += 10
    if "suspecting" in s or "illness" in s:
        score += 10

    # Pénalités pour détails secondaires souvent trompeurs
    if "laser" in s:
        score -= 3
    if "coffee" in s:
        score -= 2
    if "smooth air" in s:
        score -= 2

    # Bonus causalité / conséquence
    if any(x in s for x in ["because", "caused", "resulted", "failed", "did not", "without"]):
        score += 3

    # Bonus phrase informative
    n = len(s.split())
    if 8 <= n <= 40:
        score += 2

    return score

def select_key_sentences(text, top_k=4):
    sents = split_sentences_simple(text)
    if not sents:
        return text

    scored = [(sent, score_sentence(sent), i) for i, sent in enumerate(sents)]

    # top scoring
    top = sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

    # ajoute aussi la première phrase si absente, souvent utile pour l’événement principal
    first_sent = scored[0]
    if first_sent[2] not in [x[2] for x in top]:
        top.append(first_sent)

    # ordre original
    top = sorted(top, key=lambda x: x[2])

    # dédoublonnage par index
    seen = set()
    final = []
    for item in top:
        if item[2] not in seen:
            final.append(item[0])
            seen.add(item[2])

    return " ".join(final[:top_k])

In [79]:
def build_prompt(narrative: str) -> str:
    key_sents = select_key_sentences(narrative, top_k=4)
    return (
        "Write one short factual aviation incident synopsis using only the information below. "
        "Focus on the main incident, the main safety outcome, and the main contributing factor if stated. "
        "Do not invent a collision, injury, aircraft type, pilot role, or airport detail unless explicitly stated.\n\n"
        f"{key_sents}"
    )

In [80]:
def preprocess_function(examples):
    inputs = [build_prompt(x) for x in examples[SOURCE_COL]]
    targets = [x.strip() for x in examples["clean_target"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [81]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [82]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# Évite le warning "max_new_tokens + max_length"
model.generation_config.max_length = None

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [83]:
from datasets import DatasetDict

tokenized = DatasetDict({
    split: dataset[split].map(
        preprocess_function,
        batched=True,
        remove_columns=dataset[split].column_names,
        desc=f"Tokenizing {split}",
    )
    for split in dataset
})

print(tokenized)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})


In [84]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    predictions = np.asarray(predictions, dtype=np.int64)
    labels = np.asarray(labels, dtype=np.int64)

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    predictions = np.where(predictions < 0, pad_token_id, predictions)
    labels = np.where(labels == -100, pad_token_id, labels)
    labels = np.where(labels < 0, pad_token_id, labels)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    scores = {k: round(v * 100, 2) for k, v in scores.items()}
    return scores

In [85]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

In [86]:
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=WEIGHT_DECAY,

    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=2,

    eval_accumulation_steps=1,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
)

In [87]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [88]:
train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,14.186593,2.949180,25.670000,6.920000,20.670000,20.700000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


In [89]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/flan_t5_asrs_quick


In [90]:
def safe_postprocess(pred, narrative):
    p = pred.strip()
    n = narrative.lower()

    if "collision" in p.lower() and "collision" not in n:
        p = re.sub(r"\bcollision\b", "conflict", p, flags=re.IGNORECASE)

    if len(p) > 0:
        p = p[0].upper() + p[1:]

    return p

In [91]:
def generate_summary(narrative, max_new_tokens=32):
    prompt = build_prompt(narrative)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=6,
            no_repeat_ngram_size=3,
            length_penalty=0.8,
            early_stopping=True
        )

    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    pred = clean_prediction(pred)
    pred = safe_postprocess(pred, narrative)
    return pred

In [92]:
preds = []
refs = []

for i in range(len(dataset["test"])):
    src = dataset["test"][i][SOURCE_COL]
    ref = dataset["test"][i][TARGET_COL]

    pred = generate_summary(src, max_new_tokens=MAX_TARGET_LENGTH)

    preds.append(pred.strip())
    refs.append(ref.strip())

scores = rouge.compute(
    predictions=preds,
    references=refs,
    use_stemmer=True
)

scores = {k: round(v * 100, 2) for k, v in scores.items()}
print(scores)

Both `max_new_tokens` (=48) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

{'rouge1': np.float64(23.08), 'rouge2': np.float64(6.53), 'rougeL': np.float64(18.03), 'rougeLsum': np.float64(18.05)}


In [93]:
for i in range(5):
    ex = dataset["test"][i]
    pred = generate_summary(ex[SOURCE_COL])

    print(f"\n{'='*80}")
    print(f"EXEMPLE {i+1}")
    print(f"{'='*80}")
    print("\nNARRATIVE:\n")
    print(ex[SOURCE_COL][:1500], "..." if len(ex[SOURCE_COL]) > 1500 else "")
    print("\nREFERENCE:\n")
    print(ex[TARGET_COL])
    print("\nPREDICTION:\n")
    print(pred)

Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXEMPLE 1

NARRATIVE:

I was the Pilot monitoring on flight to ZZZ when we encountered moderate turbulence enroute at FL340 in Monterey airspace over the gulf of Mexico over the fix of XOTUG. It went from completely smooth air to moderate turbulence with a couple of jolts and rolls to where it caused several injuries to passengers and flight attendants where we had paramedics meet the aircraft upon arrival in ZZZ. We did not have any indications on RADAR nor from ATC that it was moderate level turbulence and we were completely startled ourselves. It only lasted about 10-15 seconds while the autopilot remained on and smoothed out afterwards. We checked on the cabin status and were told of the injuries incurred by the passengers who one happened to be in the lavatory and one who had coffee spilled on her lap. The Flight Attendant also hurt her wrist and hip as well from what we were told. Prior to this; the seat belt sign was off as we were experiencing smooth air and we turned it on as

Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXEMPLE 2

NARRATIVE:

JANNY5 arrival into BUR at night. Cleared to cross JANNY at and maintain 8;000 feet. SoCal Approach solicited Runway 15 visual approach (probably should have declined being at night). Crew accepted visual approach to Runway 15. We were given a descent to 6;000 ft. and a heading of 200. Turned to a heading of 200; bugged 6;000 ft. on the altitude select but failed to select a descent mode (another mistake). Cleared for the visual approach Runway 15 final at or above 3;000 ft. Crossed the mountains high and had to make a steeper than normal approach with S turns (another mistake). SoCal failed to hand the crew off to Burbank Tower (ATC error). Pilot flying was focused on the descent to get on profile for landing Runway 15. Pilot flying contemplated requesting left traffic for Runway 8 but decided to continue Runway 15 as we were getting on profile (should have requested left traffic to Runway 8). Pilot monitoring stated that a green laser light illuminated the coc

Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXEMPLE 3

NARRATIVE:

While on a visual approach to Runway 9L ATC advised us of traffic in our 10 o'clock southbound at that time we got a TCAS TA ATC instructed the traffic to turn east the traffic maintained his heading south at that point we got the RA to descend I followed the TCAS guidance and descended until clear of conflict. Once clear of the traffic we continued the approach maintaining a stabilized approach to a normal landing. The non-flying pilot did acquire the conflicting traffic as he passed approximately 150 feet above us. The conflicting traffic failed to follow ATC instructions. 

REFERENCE:

Air taxi pilot reported an NMAC with another aircraft because the other crew did not follow ATC instructions.

PREDICTION:

A non-flying aircraft encountered a conflicting traffic on a visual approach to Runway 9L.


Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EXEMPLE 4

NARRATIVE:

Aircraft X was direct for the arrival in to ZZZ. This route put the aircraft in a direct conflict with many westbound departures; so I turned the aircraft to a 080 heading to provide lateral separation. Aircraft X was level at 17;000' which was fine with me until I got it north of the westbound departure track. When the aircraft was in the vicinity of the Hinkley airport the pilot requested lower. I asked what altitude and he requested 12;000 (maybe 11;000?). There was some insistence in his tone so I coordinated with West Departures at C90 to get him lower. The controller I spoke to questioned the APREQ; but I made it clear that I thought that the pilot really did need lower. They approved 15;000' on the current heading and gave me a freq. I then shipped the aircraft to C90 and I observed the aircraft descend to 14;000'.C90 then called me back on the land line asking what was up with giving them someone low flying through their airspace. They also said that the

In [94]:
rows = []
for i in range(len(dataset["test"])):
    ex = dataset["test"][i]
    pred = generate_summary(ex[SOURCE_COL])

    rows.append({
        "narrative": ex[SOURCE_COL],
        "reference": ex[TARGET_COL],
        "prediction": pred,
    })

df_results = pd.DataFrame(rows)
csv_path = "/kaggle/working/test_predictions.csv"
df_results.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df_results.head(10)

Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

Saved: /kaggle/working/test_predictions.csv


,narrative,reference,prediction
0,I was the Pilot monitoring on flight to ZZZ wh...,B737 First Officer reported moderate turbulenc...,An aircraft encountered moderate turbulence en...
1,JANNY5 arrival into BUR at night. Cleared to c...,EMB-135 Captain reported landing without conta...,The crew of a JANNY5 was not handed off to Bur...
2,While on a visual approach to Runway 9L ATC ad...,Air taxi pilot reported an NMAC with another a...,A non-flying aircraft encountered a conflictin...
3,Aircraft X was direct for the arrival in to ZZ...,ZAU Center Controller reported an aircraft rep...,Aircraft X was directed for the arrival in to ...
4,Working satellite sectors combined; moderate w...,S46 TRACON Controller reported an aircraft bei...,A B737-700 flight crew encountered VFR traffic...
5,Early morning; approximately one hour before d...,Dispatcher reported a severe turbulence encoun...,A severe turbulence on a flight from ZZZ to ZZ...
6,Passenger in XX2 was asked 4 times to have his...,Flight Attendant reported the misconduct of a ...,A passenger in XX2 was asked 4 times to have h...
7,A VFR flight plan was active and flight follow...,Navion pilot reported diverting to the nearest...,A passenger slumped to the side and became unr...
8,We had a DH at the beginning of our day that w...,Flight Attendant reported that an old version ...,A DH was an hour and fifteen minutes late and ...
9,During climb I called ATC approach and request...,The pilot of a small airplane reported the los...,A comm radio malfunctioned during climb.


In [95]:
import re

def split_sentences_simple(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    sents = re.split(r'(?<=[\.\!\?])\s+', text)
    return [s.strip() for s in sents if s.strip()]

def score_sentence(sent):
    s = sent.lower()
    score = 0

    keywords = {
        "turbulence": 6,
        "injur": 8,
        "paramedic": 8,
        "traffic": 5,
        "conflict": 7,
        "tcas": 7,
        "ra": 6,
        "ta": 4,
        "tower": 8,
        "not handed off": 10,
        "failed to hand": 10,
        "without contacting tower": 12,
        "failed to follow atc instructions": 12,
        "separation": 8,
        "vectored": 6,
        "vector": 6,
        "localizer": 5,
        "ils": 5,
        "requested lower": 5,
        "did not want lower": 10,
        "illness": 9,
        "unsafe": 4,
    }

    for kw, w in keywords.items():
        if kw in s:
            score += w

    if any(x in s for x in ["because", "caused", "resulted", "failed", "did not", "without"]):
        score += 3

    if "laser" in s:
        score -= 4
    if "coffee" in s:
        score -= 2

    n = len(s.split())
    if 8 <= n <= 40:
        score += 2

    return score

def select_top_sentences(text, top_k=2):
    sents = split_sentences_simple(text)
    if not sents:
        return []

    scored = [(sent, score_sentence(sent), i) for i, sent in enumerate(sents)]
    top = sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]
    top = sorted(top, key=lambda x: x[2])
    return [x[0] for x in top]

In [96]:
def final_safety_cleanup(pred):
    p = pred.strip()
    p = re.sub(r"\bnon-flying aircraft\b", "another aircraft", p, flags=re.IGNORECASE)
    p = re.sub(r"\bduring departure from\b", "during approach to", p, flags=re.IGNORECASE)
    if p and not p.endswith("."):
        p += "."
    if p:
        p = p[0].upper() + p[1:]
    return p

In [97]:
def clean_text_for_summary(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\bFL\d+\b", "", text)
    text = re.sub(r"\b\d{1,2};\d{3}\b", "", text)
    text = re.sub(r"\b\d{1,2},\d{3}\b", "", text)
    text = re.sub(r"\b\d+\s*ft\.?\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bwe encountered\b", "encountered", text, flags=re.IGNORECASE)
    text = re.sub(r"\bI was\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bwe were\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bwe got\b", "received", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r"\s+\.", ".", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" ,.;")
    return text

def make_rule_based_summary(narrative):
    sents = select_top_sentences(narrative, top_k=2)
    if not sents:
        return ""

    joined = " ".join(sents)
    joined_low = joined.lower()

    if "turbulence" in joined_low and ("injur" in joined_low or "paramedic" in joined_low):
        return "Moderate turbulence resulted in injuries to passengers and flight attendants."

    if ("tower" in joined_low or "not handed off" in joined_low or "failed to hand" in joined_low):
        return "The crew landed without contacting Tower following a high-workload visual approach."

    if ("tcas" in joined_low or "ra" in joined_low or "conflict" in joined_low) and "failed to follow atc instructions" in joined_low:
        return "Conflicting traffic prompted a TCAS resolution advisory after the other aircraft failed to follow ATC instructions."

    if "requested lower" in joined_low and "did not want lower" in joined_low:
        return "An aircraft repeatedly requested lower altitudes and then denied making the request, leading controller concern about possible pilot illness."

    if ("vfr" in joined_low or "localizer" in joined_low or "ils" in joined_low) and ("vector" in joined_low or "separation" in joined_low):
        return "Conflicting VFR traffic climbing through the localizer forced ATC to vector the aircraft off the ILS approach to maintain separation."

    if (
    "requested lower" in narrative.lower()
    and (
        "did not want lower" in narrative.lower()
        or "didn't request lower" in narrative.lower()
        or "did not request lower" in narrative.lower()
        )
    ):
        return "An aircraft repeatedly requested lower altitudes and then denied making the request, leading controller concern about possible pilot illness."

    if "laser" in narr_low and ("tower" in narr_low or "not handed off" in narr_low):
        return "The crew landed without contacting Tower following a high-workload visual approach."

    if ("tcas" in narr_low or "ra" in narr_low or "ta" in narr_low) and "failed to follow atc instructions" in narr_low:
        return "Conflicting traffic prompted a TCAS resolution advisory after the other aircraft failed to follow ATC instructions."

    if ("vfr" in narr_low or "localizer" in narr_low or "ils" in narr_low) and ("vector" in narr_low or "maintain separation" in narr_low):
        return "Conflicting VFR traffic climbing through the localizer forced ATC to vector the aircraft off the ILS approach to maintain separation."

    fallback = clean_text_for_summary(sents[0])
    words = fallback.split()
    fallback = " ".join(words[:22]).strip(" ,.;")
    if fallback and not fallback.endswith("."):
        fallback += "."
    if fallback:
        fallback = fallback[0].upper() + fallback[1:]

    return fallback

In [98]:
for i in range(5):
    ex = dataset["test"][i]
    pred = make_rule_based_summary(ex[SOURCE_COL])

    print(f"\n{'='*80}")
    print(f"EXEMPLE {i+1}")
    print(f"{'='*80}")
    print("\nREFERENCE:\n")
    print(ex[TARGET_COL])
    print("\nPREDICTION:\n")
    print(pred)


EXEMPLE 1

REFERENCE:

B737 First Officer reported moderate turbulence with minor injuries to passengers and flight attendants.

PREDICTION:

Moderate turbulence resulted in injuries to passengers and flight attendants.

EXEMPLE 2

REFERENCE:

EMB-135 Captain reported landing without contacting Tower following a high workload night visual approach to BUR.

PREDICTION:

The crew landed without contacting Tower following a high-workload visual approach.

EXEMPLE 3

REFERENCE:

Air taxi pilot reported an NMAC with another aircraft because the other crew did not follow ATC instructions.

PREDICTION:

Conflicting traffic prompted a TCAS resolution advisory after the other aircraft failed to follow ATC instructions.

EXEMPLE 4

REFERENCE:

ZAU Center Controller reported an aircraft repeatedly requested lower altitudes than said they didn't request lower altitudes so the Controller descended the aircraft suspecting the pilot was suffering some sort of illness.

PREDICTION:

An aircraft repea

In [99]:
output_path = "/kaggle/working/rule_based_results_final.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_results.to_excel(writer, sheet_name="Full_Results", index=False)
    df_results[["reference", "prediction"]].to_excel(
        writer,
        sheet_name="Report_View",
        index=False
    )

print("Saved to:", output_path)

Saved to: /kaggle/working/rule_based_results_final.xlsx


In [100]:
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

output_path = "/kaggle/working/rule_based_results_final.xlsx"

wb = load_workbook(output_path)

header_fill = PatternFill(fill_type="solid", fgColor="D9EAF7")
thin = Side(style="thin", color="000000")

for ws in wb.worksheets:
    # header
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # body
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # column widths
    if ws.title == "Full_Results":
        widths = [45, 45, 120]
    else:
        widths = [50, 50]

    for i, width in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = width

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb.save(output_path)
print("Formatted:", output_path)

Formatted: /kaggle/working/rule_based_results_final.xlsx


In [102]:
import pandas as pd

metrics_rows = []

if "scores" in globals():
    for k, v in scores.items():
        metrics_rows.append({"source": "manual_rouge", "metric": k, "value": v})

if "test_metrics" in globals():
    for k, v in test_metrics.items():
        metrics_rows.append({"source": "trainer_eval", "metric": k, "value": v})

df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv("/kaggle/working/metrics_summary.csv", index=False)
df_metrics

,source,metric,value
0,manual_rouge,rouge1,23.08
1,manual_rouge,rouge2,6.53
2,manual_rouge,rougeL,18.03
3,manual_rouge,rougeLsum,18.05


In [103]:
import os
import shutil

folder_to_zip = "/kaggle/working/flan_t5_asrs_quick"
zip_base = "/kaggle/working/flan_t5_asrs_quick"

if os.path.exists(folder_to_zip):
    shutil.make_archive(zip_base, "zip", folder_to_zip)
    print("Created:", zip_base + ".zip")
else:
    print("Folder not found:", folder_to_zip)

Created: /kaggle/working/flan_t5_asrs_quick.zip
